## 1. Instalación de dependencias

In [ ]:
!pip install -U "transformers>=4.47.0" accelerate sentencepiece pillow fastapi uvicorn python-multipart pyngrok groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.3 MB/s eta 0:00:00


## 2. Configuración de credenciales (variables de entorno)

In [ ]:
import os
from google.colab import userdata

# Cargar secretos de Colab (no hardcodear claves en el notebook)
HF_TOKEN = userdata.get("HF_TOKEN")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
NGROK_TOKEN = userdata.get("NGROK_TOKEN")

if not HF_TOKEN:
    raise ValueError("Missing HF_TOKEN in Colab Secrets")
if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in Colab Secrets")
if not NGROK_TOKEN:
    raise ValueError("Missing NGROK_TOKEN in Colab Secrets")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["NGROK_AUTH_TOKEN"] = NGROK_TOKEN
print("Secrets loaded correctly.")

## Login en Hugging Face

In [ ]:
from huggingface_hub import login, whoami

# Login explícito con token para evitar 401 en repos gated
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF user:", whoami()["name"])

## Creación del backend (server.py)

In [ ]:
%%writefile server.py
import os
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from fastapi import FastAPI, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import uvicorn
import io
import tempfile
from groq import Groq

# 0. -----> CONFIGURACIÓN SEGURA
# Cargamos claves de servicios externos desde variables de entorno

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
HF_TOKEN = os.environ["HF_TOKEN"]
groq_client = Groq(api_key=GROQ_API_KEY)

# 1. -----> CARGAMOS PALIGEMMA
# Modelo multimodal visión-lenguaje (3B parámetros, resolución 448)

model_id = "google/paligemma-3b-mix-448"
device = "cuda"
dtype = torch.bfloat16

print("Cargando modelo en GPU...")

processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto",
    token=HF_TOKEN,
).eval()

print("Modelo cargado correctamente en GPU")

# 2. -----> FASTAPI
# Backend que expone los endpoints del sistema

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 3. -----> TRANSCRIBE (WHISPER - GROQ)
# Endpoint POST /transcribe
# Domain prompt steers Whisper towards campus navigation vocabulary.

WHISPER_DOMAIN_PROMPT = (
    "UFV campus navigation assistant. "
    "classroom 0.1, classroom 0.3, classroom 1.1, classroom 1.4, classroom 2.1, "
    "building G, building CD, building H, main entrance, ground floor, first floor, second floor, "
    "elevator, stairs, cafeteria, gym, restroom, AI laboratory, mentoring room, "
    "EPS faculty offices, secretary, directory sign, "
    "I want to go to, navigate to, take me to, I am in, "
    "yes, no, start, done, repeat, cancel, I am lost, I am in a hurry."
)

@app.post("/transcribe")
async def transcribe(file: UploadFile):
    audio_bytes = await file.read()
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        tmp.write(audio_bytes)
        audio_path = tmp.name

    with open(audio_path, "rb") as audio_file:
        result = groq_client.audio.transcriptions.create(
            file=audio_file,
            model="whisper-large-v3",
            response_format="text",
            language="en",
            temperature=0.0,
            prompt=WHISPER_DOMAIN_PROMPT,
        )

    os.unlink(audio_path)
    text = result.strip() if isinstance(result, str) else ""
    print(f"[TRANSCRIBE] {text!r}")
    return {"success": True, "text": text}

# 4. -----> ASK (PALIGEMMA)
# Endpoint POST /ask

@app.post("/ask")
async def ask(file: UploadFile, text: str = Form(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    inputs = processor(text=text, images=image, return_tensors="pt").to(device, dtype=dtype)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=300, do_sample=False)

    answer = processor.decode(output[0][input_len:], skip_special_tokens=True)
    return {"answer": answer}

Writing server.py


## Configuración de ngrok

In [ ]:
from pyngrok import ngrok

# Always close previous tunnels to avoid ERR_NGROK_334.
ngrok.kill()
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])

public_url = ngrok.connect(8000)
public_url

<NgrokTunnel: "https://your-subdomain.ngrok-free.app" -> "http://localhost:8000">

## Ejecución de Uvicorn

In [ ]:
!uvicorn server:app --host 0.0.0.0 --port 8000

2025-12-19 13:54:33.025916: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766152473.045754    2263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766152473.051721    2263 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766152473.066547    2263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766152473.066571    2263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766152473.066575    2263 computation_placer.cc:177] computation placer alr

In [ ]:
print(public_url)
print(public_url.public_url)

NgrokTunnel: "https://your-subdomain.ngrok-free.app" -> "http://localhost:8000"
https://your-subdomain.ngrok-free.app
